# LSTM: controlled memory

**Learning objective:** Understand the cell state and gates, then inspect TensorFlow's LSTM shapes and parameterization.

This notebook is part of the TensorFlow/Keras learning track. It is designed to be read top-to-bottom: intuition → shapes → mathematics → TensorFlow implementation → observed result → interpretation.

> GitHub renders the committed executed output as a static learning artifact. Clone the repository and rerun it in Jupyter/VS Code for live experimentation.


In [1]:
import os, warnings, random
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "bundled with TensorFlow")
print("Execution device(s):", [d.device_type for d in tf.config.list_logical_devices()])


2026-09-21 07:13:37.781030: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789974817.795805    3043 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789974817.800052    3043 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.1
Keras: 3.15.1
Execution device(s): ['CPU']


2026-09-21 07:13:39.429834: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


An LSTM learns four transformations: forget gate, input gate, candidate content and output gate. The cell state provides a controlled memory path:
        \[c_t=f_t\odot c_{t-1}+i_t\odot\tilde{c}_t,\quad h_t=o_t\odot\tanh(c_t).\]


In [2]:
sigmoid=lambda z:1/(1+np.exp(-z))
c_prev=np.array([0.8,-0.4]); f=sigmoid(np.array([2.0,-1.0])); i=sigmoid(np.array([-0.5,1.5])); candidate=np.tanh(np.array([0.3,0.9])); o=sigmoid(np.array([1.0,0.2]))
c=f*c_prev+i*candidate; h=o*np.tanh(c)
display(pd.DataFrame({"c_prev":c_prev,"forget":f,"input":i,"candidate":candidate,"c_new":c,"output":o,"h_new":h}).round(4))


,c_prev,forget,input,candidate,c_new,output,h_new
0,0.8,0.8808,0.3775,0.2913,0.8146,0.7311,0.4914
1,-0.4,0.2689,0.8176,0.7163,0.4781,0.5498,0.2445


In [3]:
x=tf.random.normal((4,10,3),seed=SEED)
lstm=tf.keras.layers.LSTM(6,return_sequences=True,return_state=True)
seq,h,c=lstm(x)
print("sequence output:",seq.shape,"hidden state:",h.shape,"cell state:",c.shape)
print("parameters:",lstm.count_params())


sequence output: (4, 10, 6) hidden state: (4, 6) cell state: (4, 6)
parameters: 240


Compared with a vanilla RNN, an LSTM carries both `h` (exposed hidden representation) and `c` (cell memory).
